In [68]:
import os
import pandas as pd
import numpy as np


# now dump the data to the pg database
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine

In [69]:
file_name = 'extended/wanderers.csv'
print(f"Cargando datos... {file_name}")

# current_rival = get_rival_name(file_name)
# file = 'partidos/' + file_name
file = 'work/cargar_datos_base/partidos/' + file_name
# column_names = [
#     "Player Id", "Player Display Name", "Time", "Elapsed Time (s)",
#     "Lat", "Lon", "Speed km/h", "Instantaneous Acceleration Impulse", "Heart Rate (bpm)", ""
# ]


# Player Id, Player Display Name, Session Id, Time, Elapased Time, Speed (m/s), Instantaneous Acceleration Impulse, Heart Rate Interval, Latitude, Longitude,  Accl X, Accl Y, Accl Z, Gyro X, Gyro Y, Gyro Z, Hacc, Hdop, Quality of Signal, No. of Satellites
columnas_originales = [
        "Player Id", "Player Display Name", "Time", "Elapsed Time (s)",
        "Lat", "Lon", "Speed km/h", "Instantaneous Acceleration Impulse", "Heart Rate (bpm)", ""
]
datos_csv = pd.read_csv(file, delimiter=',', skiprows=1, encoding='latin1', names=columnas_originales)

Cargando datos... extended/wanderers.csv


In [70]:
datos_csv.head()

# if not extended
new_df = datos_csv


In [71]:
# # now reformat the data, so it follows this schema
# # column_names = ["Player Id", "Player Display Name", "Time", "Elapsed Time (s)","Lat", "Lon", "Speed km/h", "Instantaneous Acceleration Impulse", "Heart Rate (bpm)", ""]
# # fix speed to km/h, and only keep two decimals

# new_df = pd.DataFrame()
# new_df["Player Id"] = datos_csv['Player Id']
# new_df["Player Display Name"] = datos_csv['Player Display Name']
# new_df["Time"] = datos_csv['Time']
# new_df["Elapsed Time (s)"] = datos_csv['Elapsed Time']
# new_df["Lat"] = datos_csv['Latitude']
# new_df["Lon"] = datos_csv['Longitude']
# new_df["Speed km/h"] = datos_csv['Speed (m/s)'] * 3.6
# new_df["Speed km/h"] = new_df["Speed km/h"].apply(lambda x: round(x, 2))
# new_df["Instantaneous Acceleration Impulse"] = datos_csv['Instantaneous Acceleration Impulse']
# new_df["Heart Rate (bpm)"] = datos_csv['Heart Rate Interval']

# new_df.head()


In [72]:
# top 10 max speed, check if the data is correct

top_10_max_speed = new_df.sort_values(by="Speed km/h", ascending=False).head(10)
top_10_max_speed

Player Id  \
68 Del Pino Esquivo 1866 08:57:38.02 0.019998   0.719445 0.638889 0 -34.853875 -56.311395     0.2500   
82 Calione Fuentes  1866 08:56:25.57 249.079130 0.000000 0.000000 0 -34.854133 -56.311644    -0.1000   
                         08:56:25.55 249.059130 0.000000 0.000000 0 -34.854133 -56.311644    -0.0500   
                         08:56:25.54 249.049130 0.000000 0.000000 0 -34.854134 -56.311644    -0.0250   
                         08:56:25.53 249.039130 0.000000 0.000000 0 -34.854134 -56.311644    -0.0125   
                         08:56:25.52 249.029130 0.000000 0.000000 0 -34.854134 -56.311644    -0.0125   
                         08:56:25.51 249.019130 0.000000 0.000000 0 -34.854134 -56.311644    -0.0500   
                         08:56:25.50 249.009130 0.000000 0.000000 0 -34.854134 -56.311644    -0.1000   
                         08:56:25.49 248.999130 0.000000 0.000000 0 -34.854134 -56.311644    -0.1000   
                         08:56:25.48 248.989130 0.000000 0.000000 0 -34.854134 -56.311644    -0.0875   

                                                                                           Player Display Name  \
68 Del Pino Esquivo 1866 08:57:38.02 0.019998   0.719445 0.638889 0 -34.853875 -56.311395               0.8250   
82 Calione Fuentes  1866 08:56:25.57 249.079130 0.000000 0.000000 0 -34.854133 -56.311644               0.4000   
                         08:56:25.55 249.059130 0.000000 0.000000 0 -34.854133 -56.311644               0.4750   
                         08:56:25.54 249.049130 0.000000 0.000000 0 -34.854134 -56.311644               0.5125   
                         08:56:25.53 249.039130 0.000000 0.000000 0 -34.854134 -56.311644               0.5125   
                         08:56:25.52 249.029130 0.000000 0.000000 0 -34.854134 -56.311644               0.4750   
                         08:56:25.51 249.019130 0.000000 0.000000 0 -34.854134 -56.311644               0.4375   
                         08:56:25.50 249.009130 0.000000 0.000000 0 -34.854134 -56.311644               0.4125   
                         08:56:25.49 248.999130 0.000000 0.000000 0 -34.854134 -56.311644               0.4625   
                         08:56:25.48 248.989130 0.000000 0.000000 0 -34.854134 -56.311644               0.4875   

                                                                                             Time  \
68 Del Pino Esquivo 1866 08:57:38.02 0.019998   0.719445 0.638889 0 -34.853875 -56.311395  0.4500   
82 Calione Fuentes  1866 08:56:25.57 249.079130 0.000000 0.000000 0 -34.854133 -56.311644  0.9500   
                         08:56:25.55 249.059130 0.000000 0.000000 0 -34.854133 -56.311644  1.0000   
                         08:56:25.54 249.049130 0.000000 0.000000 0 -34.854134 -56.311644  0.9625   
                         08:56:25.53 249.039130 0.000000 0.000000 0 -34.854134 -56.311644  0.9125   
                         08:56:25.52 249.029130 0.000000 0.000000 0 -34.854134 -56.311644  0.9125   
                         08:56:25.51 249.019130 0.000000 0.000000 0 -34.854134 -56.311644  0.9250   
                         08:56:25.50 249.009130 0.000000 0.000000 0 -34.854134 -56.311644  0.9500   
                         08:56:25.49 248.999130 0.000000 0.000000 0 -34.854134 -56.311644  0.9000   
                         08:56:25.48 248.989130 0.000000 0.000000 0 -34.854134 -56.311644  0.9000   

                                                                                           Elapsed Time (s)  \
68 Del Pino Esquivo 1866 08:57:38.02 0.019998   0.719445 0.638889 0 -34.853875 -56.311395              51.2   
82 Calione Fuentes  1866 08:56:25.57 249.079130 0.000000 0.000000 0 -34.854133 -56.311644             102.4   
                         08:56:25.55 249.059130 0.000000 0.000000 0 -34.854133 -56.311644              51.2   
                         08:56:25.54 249.049130 0.000000 0.000000 0 -34.854134 -56.311644               0.0   
                        

In [73]:
# show max and min elapsed time by player
max_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].max()
min_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].min()
display(max_elapsed_time)
display(min_elapsed_time)

Player Display Name
-13.4000   -6912.0
-13.1250   -4608.0
-11.2125   -1433.6
-10.1625    2048.0
-10.0000   -5529.6
             ...  
 16.0125    3840.0
 16.7375   -3532.8
 16.9000    2560.0
 17.3375    -716.8
 25.5750    2662.4
Name: Elapsed Time (s), Length: 1407, dtype: float64

Player Display Name
-13.4000   -6912.0
-13.1250   -4608.0
-11.2125   -1433.6
-10.1625    2048.0
-10.0000   -5529.6
             ...  
 16.0125    3840.0
 16.7375   -3532.8
 16.9000    2560.0
 17.3375    -716.8
 25.5750   -1536.0
Name: Elapsed Time (s), Length: 1407, dtype: float64

In [74]:
# now show the max and min times for each player
max_time = new_df.groupby("Player Display Name")["Time"].max()
min_time = new_df.groupby("Player Display Name")["Time"].min()
display(max_time)
display(min_time)


Player Display Name
-13.4000    14.2000
-13.1250   -19.7375
-11.2125     2.3250
-10.1625    -3.6000
-10.0000    -4.6125
             ...   
 16.0125     6.3750
 16.7375    10.2250
 16.9000     9.3125
 17.3375     7.0375
 25.5750     7.9000
Name: Time, Length: 1407, dtype: float64

Player Display Name
-13.4000    14.2000
-13.1250   -19.7375
-11.2125     2.3250
-10.1625    -3.6000
-10.0000    -4.6125
             ...   
 16.0125     6.3750
 16.7375    10.2250
 16.9000     9.3125
 17.3375     7.0375
 25.5750     6.5500
Name: Time, Length: 1407, dtype: float64

In [75]:
# fix the elapsed times, there are player who have a minimum time near 0, which is ok, but there are other players who have a minimum time of 57000 seconds, which is not ok,
# so we will fix this by substracting the minimum time to all the times of the player who have a minimum time greater than 1

min_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].min()
players_to_fix = min_elapsed_time[min_elapsed_time > 1].index

# Convert the "Time" column to datetime format
min_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].min()

print(type(min_elapsed_time[0])) # <class 'numpy.float64'>

# remove the miliseconds, and leave only the seconds, don't modify the type, it has to be a float
min_elapsed_time = min_elapsed_time.apply(lambda x: int(x))

for player in players_to_fix:    
    new_df.loc[new_df["Player Display Name"] == player, "Elapsed Time (s)"] = new_df.loc[new_df["Player Display Name"] == player, "Elapsed Time (s)"] - min_elapsed_time[player]    
    

max_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].max()
display(max_elapsed_time)

<class 'numpy.float64'>


Player Display Name
-13.4000   -6912.0
-13.1250   -4608.0
-11.2125   -1433.6
-10.1625       0.0
-10.0000   -5529.6
             ...  
 16.0125       0.0
 16.7375   -3532.8
 16.9000       0.0
 17.3375    -716.8
 25.5750    2662.4
Name: Elapsed Time (s), Length: 1407, dtype: float64

In [76]:
# sort data by player id and elapsed time
new_df = new_df.sort_values(by=['Player Id', 'Elapsed Time (s)'])

In [77]:
# save the data to a csv file
new_file_name = file.replace('extended/', 'cleaned/')

# persist the data to a csv file
new_df.to_csv(new_file_name, index=False)
